# Stance Comparison: Ordinal DCM Across All Theories

Loads cached results from `run_all_stances.py` and compares $P(\text{consciousness} = 1 \mid \text{data})$ across all 13 stances for the target system (2024 Leading Chat LLMs).

**Prerequisite:** Run `python run_all_stances.py` (or `--quick` for draft results) to populate `results/`.

In [ ]:
import sys, os, json, warnings
from pathlib import Path

sys.path.insert(0, "..")
os.chdir(os.path.join(os.path.dirname(os.path.abspath("."))))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import arviz as az
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

RESULTS_DIR = Path("results")

---
## 1. Load cached results

In [ ]:
# Load summary JSON
summary_path = RESULTS_DIR / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        f"{summary_path} not found. Run: python run_all_stances.py"
    )

with open(summary_path) as f:
    summaries = json.load(f)

print(f"Loaded results for {len(summaries)} stances")
print(f"{'Stance':<45} {'Indicators':>10} {'Observations':>12} {'Time':>6}")
print("-" * 80)
for s in summaries:
    print(
        f"{s['stance']:<45} {s['n_indicators']:>10} "
        f"{s['n_observations']:>12} {s['elapsed_s']:>5.0f}s"
    )

In [ ]:
# Load individual idata files for deeper analysis
def sanitise_filename(name: str) -> str:
    return name.lower().replace(" ", "_").replace("(", "").replace(")", "")

idata_cache = {}
for s in summaries:
    nc_path = RESULTS_DIR / f"{sanitise_filename(s['stance'])}.nc"
    if nc_path.exists():
        idata_cache[s["stance"]] = az.from_netcdf(str(nc_path))

print(f"Loaded {len(idata_cache)} idata files from disk")

---
## 2. Forest plot: P(consciousness = 1) across stances

The headline comparison. Each row is one stance (theory of consciousness); the point is the posterior mean and the bar is the 94% credible interval. Dashed line marks the prior mean under $\text{Beta}(1, 5)$.

In [ ]:
# Sort by posterior mean
sorted_s = sorted(summaries, key=lambda s: s["p_consciousness_mean"])

names = [s["stance"] for s in sorted_s]
means = [s["p_consciousness_mean"] for s in sorted_s]
lows = [s["p_consciousness_ci_low"] for s in sorted_s]
highs = [s["p_consciousness_ci_high"] for s in sorted_s]
y = np.arange(len(names))

fig, ax = plt.subplots(figsize=(8, max(5, 0.45 * len(names))))
ax.barh(y, means, color="steelblue", alpha=0.7, height=0.6, edgecolor="white")
ax.errorbar(
    means, y,
    xerr=[np.array(means) - lows, np.array(highs) - means],
    fmt="none", color="black", capsize=3, linewidth=1,
)
ax.axvline(1 / 6, color="grey", linestyle="--", alpha=0.6, label="Prior mean (1/6)")
ax.set_yticks(y)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("P(consciousness = 1 | data)")
ax.set_title("Posterior probability of consciousness across stances\n(2024 Leading Chat LLMs)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

# Numeric table
print(f"{'Stance':<45} {'P(C=1)':>7} {'94% CI':>16}")
print("-" * 72)
for s in sorted(summaries, key=lambda s: s["p_consciousness_mean"], reverse=True):
    print(
        f"{s['stance']:<45} {s['p_consciousness_mean']:>7.3f} "
        f"[{s['p_consciousness_ci_low']:.3f}, {s['p_consciousness_ci_high']:.3f}]"
    )

---
## 3. Convergence diagnostics across stances

In [ ]:
print(f"{'Stance':<45} {'Max Rhat':>9} {'Min ESS':>9} {'Status':>8}")
print("-" * 75)
n_warnings = 0
for s in summaries:
    rhat_ok = s["max_rhat"] < 1.01
    ess_ok = s["min_ess"] > 100
    status = "OK" if (rhat_ok and ess_ok) else "WARNING"
    if status == "WARNING":
        n_warnings += 1
    print(
        f"{s['stance']:<45} {s['max_rhat']:>9.4f} {s['min_ess']:>9.0f} {status:>8}"
    )

print(f"\n{n_warnings} stance(s) with convergence warnings")

---
## 4. Observation model parameters across stances

Since all stances share the same 4 experts, comparing the estimated expert shifts $b_e$ and discrimination $a$ across stances reveals whether the observation model behaves consistently.

In [ ]:
# Extract observation model parameters from each stance's idata
obs_params = []
for s in summaries:
    name = s["stance"]
    idata = idata_cache.get(name)
    if idata is None:
        continue
    post = idata.posterior
    a_mean = float(post["a"].mean())
    kappa_mean = post["kappa"].mean(dim=("chain", "draw")).values

    entry = {
        "stance": name,
        "a": a_mean,
        "kappa": kappa_mean.tolist(),
        "anchor": s["anchor_expert"],
    }

    if "b_free" in post:
        b_free = post["b_free"].mean(dim=("chain", "draw")).values
        entry["b_free"] = b_free.tolist()
    else:
        entry["b_free"] = []

    obs_params.append(entry)

# Discrimination comparison
fig, axes = plt.subplots(1, 2, figsize=(12, max(4, 0.4 * len(obs_params))))

# Panel 1: Discrimination (a)
ax = axes[0]
stance_names = [p["stance"] for p in obs_params]
a_vals = [p["a"] for p in obs_params]
y = np.arange(len(obs_params))
ax.barh(y, a_vals, color="steelblue", alpha=0.7, height=0.6)
ax.set_yticks(y)
ax.set_yticklabels(stance_names, fontsize=8)
ax.set_xlabel("Discrimination (a)")
ax.set_title("Discrimination across stances")

# Panel 2: Cutpoints
ax = axes[1]
for i, p in enumerate(obs_params):
    kappa = p["kappa"]
    ax.scatter(kappa, [i] * len(kappa), alpha=0.6, s=20, color="steelblue")
ax.set_yticks(y)
ax.set_yticklabels(stance_names, fontsize=8)
ax.set_xlabel("Cutpoint value")
ax.set_title("Cutpoints across stances")

fig.tight_layout()
plt.show()

# Print table
print(f"{'Stance':<45} {'a':>6} {'Anchor':<20}")
print("-" * 75)
for p in obs_params:
    print(f"{p['stance']:<45} {p['a']:>6.2f} {p['anchor']:<20}")

---
## 5. Prior-domination check across stances

For each stance, compare the posterior mean of $P(\text{consciousness}=1)$ to the $\text{Beta}(1,5)$ prior mean (0.167). If posteriors cluster near the prior, the model is prior-dominated everywhere.

In [ ]:
prior_mean = 1 / 6  # Beta(1, 5)

fig, ax = plt.subplots(figsize=(6, 4))
post_means = [s["p_consciousness_mean"] for s in summaries]
n_indicators = [s["n_indicators"] for s in summaries]

ax.scatter(n_indicators, post_means, s=60, alpha=0.7, color="steelblue", edgecolor="white")
for i, s in enumerate(summaries):
    # Abbreviate long names
    short = s["stance"][:20]
    ax.annotate(
        short, (n_indicators[i], post_means[i]),
        textcoords="offset points", xytext=(5, 3), fontsize=7, alpha=0.7,
    )

ax.axhline(prior_mean, color="salmon", linestyle="--", alpha=0.7, label=f"Prior mean ({prior_mean:.3f})")
ax.set_xlabel("Number of indicators with observations")
ax.set_ylabel("P(consciousness = 1 | data)")
ax.set_title("Posterior vs number of indicators")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

shifts = [abs(m - prior_mean) for m in post_means]
print(f"Mean |shift from prior|: {np.mean(shifts):.4f}")
print(f"Max  |shift from prior|: {np.max(shifts):.4f}")
print(f"Stance with largest shift: {summaries[np.argmax(shifts)]['stance']}")

---
## 6. Posterior distributions (violin plot)

In [ ]:
# Collect posterior draws for each stance
from dcm_model import ModelConfig

stance_draws = {}
for s in summaries:
    idata = idata_cache.get(s["stance"])
    if idata is None:
        continue
    # Find stance beta variable (first variable ending in _beta that matches)
    for vname in idata.posterior.data_vars:
        if vname.endswith("_beta") and vname != "a":
            # Check it is the stance-level variable (scalar, not array)
            var = idata.posterior[vname]
            if var.dims == ("chain", "draw"):
                stance_draws[s["stance"]] = var.values.flatten()
                break

# Sort by mean
sorted_names = sorted(stance_draws.keys(), key=lambda n: stance_draws[n].mean())

fig, ax = plt.subplots(figsize=(8, max(5, 0.5 * len(sorted_names))))
parts = ax.violinplot(
    [stance_draws[n] for n in sorted_names],
    positions=range(len(sorted_names)),
    vert=False,
    showmeans=True,
    showmedians=False,
)
for pc in parts["bodies"]:
    pc.set_facecolor("steelblue")
    pc.set_alpha(0.6)
parts["cmeans"].set_color("black")

ax.axvline(prior_mean, color="salmon", linestyle="--", alpha=0.7, label="Prior mean")
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("P(consciousness = 1 | data)")
ax.set_title("Posterior distributions across stances")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

---
## 7. Summary

Key observations to discuss with Arvo:

1. **Cross-stance variation:** How much do the posteriors vary across theories? If they all cluster near the prior mean, the support/demandingness priors dominate universally. If there is meaningful spread, some theories are more data-sensitive than others.

2. **Convergence:** Any stances with $\hat{R} > 1.01$ or ESS < 100 need further investigation (longer tuning, reparameterisation, or data issues).

3. **Observation model consistency:** If discrimination $a$ and cutpoints $\kappa$ are similar across stances, this validates the shared observation model. If they differ substantially, it may indicate stance-specific response patterns.

4. **Anchor expert variation:** If different stances select different anchor experts, the expert shifts $b_e$ are not directly comparable across stances. Worth standardising if cross-stance expert comparisons are needed.